# Forecasting Antimicrobial Resistance in the EU/EEA**QM640 Data Analytics Capstone — Walsh College**All paths are absolute under `/content/`, the Colab working directory.Run the cells **in order**; Cell 1 must run first (and again after any runtime restart).

## 1 · Setup — paths, dependencies, project files

In [ ]:
# ============================================================# CELL 1 — Setup. Run first, and again after any restart.# ============================================================import os, sys, glob, subprocess# ---------- absolute Colab paths ----------REPO_DIR    = '/content/repo'                      # project codeDATA_DIR    = os.path.join(REPO_DIR, 'data')       # raw public dataSRC_DIR = os.path.join(REPO_DIR, 'src')    # pipeline scriptsOUT_DIR     = '/content/outputs'                   # generated tablesFIGS_DIR    = '/content/figures'                   # generated figuresfor d in (OUT_DIR, FIGS_DIR):    os.makedirs(d, exist_ok=True)# scripts read these at import timeos.environ['PROJECT_ROOT'] = REPO_DIRos.environ['DATA_DIR']     = DATA_DIRos.environ['OUT_DIR']      = OUT_DIRos.environ['FIGS_DIR']     = FIGS_DIRIN_COLAB = 'google.colab' in sys.modulesprint('Colab:', IN_COLAB)# ---------- dependencies ----------!pip install -q openpyxl statsmodels# ---------- get the project ----------USE_UPLOAD = False                                  # True -> upload Capstone.zip insteadREPO_URL   = 'https://github.com/shramik-dev/Capstone.git'if not os.path.exists(SRC_DIR):    if USE_UPLOAD:        from google.colab import files        print('Upload Capstone.zip ...')        up = files.upload()        !unzip -qo "/content/{list(up.keys())[0]}" -d /content/        !mv /content/Capstone {REPO_DIR}    else:        !git clone -q {REPO_URL} {REPO_DIR} || echo 'Clone failed - set USE_UPLOAD = True and re-run.'sys.path.insert(0, SRC_DIR)# ---------- verify ----------need = ['ears_combined.csv', 'consumption_ALL_with_Germany.csv',        'nrg_chdd_a_defaultview_spreadsheet.xlsx',        'demo_r_d3dens__custom_22308386_spreadsheet.xlsx']missing = [f for f in need if not os.path.exists(os.path.join(DATA_DIR, f))]if missing:    print('\nMISSING data files:', missing)    print('Set USE_UPLOAD = True above and re-run this cell.')else:    print('\nReady. Paths in use:')    for k in ('PROJECT_ROOT','DATA_DIR','OUT_DIR','FIGS_DIR'):        print(f'  {k:13s} {os.environ[k]}')# ---------- helper: show saved figures inline ----------from IPython.display import Image, displaydef show_figs(subdir, title=None):    folder = os.path.join(FIGS_DIR, subdir)    files = sorted(glob.glob(os.path.join(folder, '*.png')))    if title:        print(f'\n===== {title}  ({len(files)} figures) =====')    for f in files:        print('\n' + os.path.basename(f))        display(Image(filename=f))

## 2 · Build the merged analytical panel

In [ ]:
# ============================================================# CELL 2 — Build the panel  ->  /content/outputs/amr_panel.csv# ============================================================!python {SRC_DIR}/data_prep.pyimport pandas as pdpanel = pd.read_csv(os.path.join(OUT_DIR, 'amr_panel.csv'))print('\nShape:', panel.shape)panel.head()

## 3 · Exploratory data analysis

In [ ]:
# ============================================================# CELL 3 — EDA  ->  /content/figures/eda/      (~1 min)# ============================================================!python {SRC_DIR}/01_eda.pyshow_figs('eda', 'EDA figures')

## 4 · Hypothesis tests and power analysis

In [ ]:
# ============================================================# CELL 4 — Shapiro-Wilk, ANOVA, Kruskal-Wallis, Spearman, power# ============================================================!python {SRC_DIR}/stats.py

## 5 · Modelling — RQ1 to RQ4**Slow cell** — grid search over five combinations takes several minutes.

In [ ]:
# ============================================================# CELL 5 — Modelling  ->  /content/figures/model/   (SLOW)# ============================================================!python {SRC_DIR}/02_modeling.pyshow_figs('model', 'Model figures')

## 6 · Unified model comparisonAll four models, one comparable experiment. Also slow.

In [ ]:
# ============================================================# CELL 6 — Unified comparison  (SLOW)# ============================================================!python {SRC_DIR}/unified.pyimport pandas as pdf = os.path.join(OUT_DIR, 'unified_results.csv')if os.path.exists(f):    display(pd.read_csv(f, index_col=0).round(3))

## 7 · Supporting analysesPer-RQ sample size, IQR/kurtosis, paired model significance tests.

In [ ]:
# ============================================================# CELL 7 — Supporting analyses  (SLOW)# ============================================================!python {SRC_DIR}/gaps.py

## 8 · Download results

In [ ]:
# ============================================================# CELL 8 — Zip figures + outputs and download# ============================================================import zipfileZIP = '/content/amr_results.zip'with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as z:    for base in (FIGS_DIR, OUT_DIR):        for root, _, fs in os.walk(base):            for f in fs:                full = os.path.join(root, f)                z.write(full, os.path.relpath(full, '/content'))print('Archive:', ZIP, round(os.path.getsize(ZIP)/1e6, 2), 'MB')if IN_COLAB:    from google.colab import files    files.download(ZIP)